In [1]:
import json
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, matplotlib.pyplot as plt, shap, joblib, thermoift.PLOT_SETTINGS as ps
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator, LogLocator
from sklearn.model_selection import train_test_split, cross_validate, KFold
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from thermoift.rng_utils import get_rng
from thermoift import MLPostprocessing, plot_correlation_heatmap, print_model_metrics

# Respect SLURM CPU allocation — prevents all 254 node CPUs being grabbed
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")

Thread limit set to 4 (SLURM_CPUS_PER_TASK)


In [2]:
OUTPUT_FOLDER = "XGBGAMMA_OUTPUTS"
SEED          = 455552
TEST_ROWS     = None
DATA_PATH     = ""

In [3]:
# Parameters
OUTPUT_FOLDER = "/scratch-shared/draju/PART_2/RANDOM/XGB_OUTPUTS/N025/trial_18/XGBGamma"
TEST_ROWS = None
SEED = 50023
DATA_PATH = "/scratch-shared/draju/PART_2/RANDOM/COMBINED/N025/trial_18.csv"


In [4]:
df = pd.read_csv(DATA_PATH)

if isinstance(TEST_ROWS, str) and TEST_ROWS.strip().lower() in ("", "none", "null"):
    TEST_ROWS = None
if TEST_ROWS is not None:
    TEST_ROWS = int(TEST_ROWS)
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")

print(f"Total samples: {len(df)}")
print(f"\ngamma statistics:")
print(df["gamma"].describe())

Total samples: 4856

gamma statistics:
count    4856.000000
mean        9.989973
std         6.168485
min         0.051914
25%         4.508454
50%         9.577015
75%        15.225555
max        22.533452
Name: gamma, dtype: float64


In [5]:
target     = "gamma"
rng        = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split: first 70/30, then split 30 into 15/15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   3399
Testing samples:    728
Validation samples: 729


In [6]:
# XGBoost Regressor
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=rng,
    n_jobs=n_cpus)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_train_pred = xgb_model.predict(X_train)
y_test_pred  = xgb_model.predict(X_test)
y_val_pred   = xgb_model.predict(X_val)

# Metrics
metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="mN/m", y_val=y_val, y_val_pred=y_val_pred)

Model Performance for gamma

Training Set:
  R²:   0.999935
  RMSE: 0.049897 mN/m
  MAE:  0.032204 mN/m

Test Set:
  R²:   0.999731
  RMSE: 0.100170 mN/m
  MAE:  0.066237 mN/m

Validation Set:
  R²:   0.999722
  RMSE: 0.103290 mN/m
  MAE:  0.063450 mN/m


In [7]:
results_df = pd.DataFrame({
    "idx": np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual": np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred, y_test_pred, y_val_pred]),
    "split": ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(OUTPUT_FOLDER, f"XGB_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

# Save fitted model for future use without retraining
model_path = os.path.join(OUTPUT_FOLDER, f"XGB_{target}_model.joblib")
joblib.dump(xgb_model, model_path)
print(f"Model saved to: {model_path}")

Predictions saved: 4856 rows
Model saved to: /scratch-shared/draju/PART_2/RANDOM/XGB_OUTPUTS/N025/trial_18/XGBGamma/XGB_gamma_model.joblib


In [8]:
# 5-fold cross-validation — single pass, all 4 metrics (5 fits instead of 20)
# n_jobs=1: XGBRegressor already parallelizes internally; loky workers fight over /dev/shm
cv_results = cross_validate(
    xgb_model, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
        "mape": "neg_mean_absolute_percentage_error",
    },
    n_jobs=1,
)
cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]
cv_mape_scores = -cv_results["test_mape"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAPE Scores: {cv_mape_scores * 100}")
print(f"Mean CV MAPE: {cv_mape_scores.mean() * 100:.4f}% (+/- {cv_mape_scores.std() * 2 * 100:.4f}%)")

Cross-Validation R² Scores:   [0.99690633 0.99963751 0.99609615 0.99456262 0.99260845]
Mean CV R²:   0.995962 (+/- 0.004701)

Cross-Validation RMSE Scores: [0.33317646 0.12102448 0.39266789 0.4508595  0.517604  ]
Mean CV RMSE: 0.363066 (+/- 0.271210)

Cross-Validation MAE Scores:  [0.17725807 0.08838429 0.2583165  0.28193208 0.36027733]
Mean CV MAE:  0.233234 (+/- 0.186022)

Cross-Validation MAPE Scores: [2.33482735 1.28406231 3.73547919 5.80045935 7.5745021 ]
Mean CV MAPE: 4.1459% (+/- 4.5722%)


In [9]:
metrics["cv_r2_scores"]   = cv_r2_scores.tolist()
metrics["cv_r2_mean"]     = float(cv_r2_scores.mean())
metrics["cv_r2_std"]      = float(cv_r2_scores.std())
metrics["cv_rmse_scores"] = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]   = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]    = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]  = cv_mae_scores.tolist()
metrics["cv_mae_mean"]    = float(cv_mae_scores.mean())
metrics["cv_mae_std"]     = float(cv_mae_scores.std())
metrics["cv_mape_scores"] = (cv_mape_scores * 100).tolist()
metrics["cv_mape_mean"]   = float(cv_mape_scores.mean() * 100)
metrics["cv_mape_std"]    = float(cv_mape_scores.std() * 100)
metrics["model"]          = "XGBoost"
metrics["features"]       = list(features)
metrics["target"]         = target
metrics["seed"]           = SEED

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
metrics_path = os.path.join(OUTPUT_FOLDER, f"XGB_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Metrics saved to: {metrics_path}")

Metrics saved to: /scratch-shared/draju/PART_2/RANDOM/XGB_OUTPUTS/N025/trial_18/XGBGamma/XGB_gamma_metrics.json


In [10]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 0.14 minutes
